# SPY 0DTE Opening Range Breakout — Quick Runner

Strategy in one line: SPY breaks the first-N-min opening range → buy ATM 0DTE call (long) or put (short) → exit at 5% net profit, 10% net stop, or 30 min time stop (defaults; all sweepable).

Run cells top to bottom with the ▶ button. No editing needed.

## 1. Setup (just clone the repo)

In [ ]:
import os, sys
print('a) preparing...')
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
if not os.path.isdir(REPO):
    print('b) cloning repo (a few seconds)...')
    !git clone --quiet -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO}
else:
    print('b) repo exists; pulling latest...')
    !cd {REPO} && git pull --quiet
print('c) adding src/ to PYTHONPATH...')
SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Run the sweep (648 configs, ~15 min)

Sweep dimensions:
- **OR windows:** 5, 15, 30 min
- **Confluence:** none (any break) or any (must clear PDH/PMH/ONH on longs; or the inverse on shorts)
- **Profit target:** 3 / 5 / 7 / 10 % net of all fees
- **Stop loss:** 5 / 10 / 15 % net
- **Time stop:** 15 / 30 / 60 min
- **Entry cutoff:** 10:30 / 11:00 / 11:30 ET

First run on a fresh runtime will be slower (Polygon data cache is cold; needs to fetch 30 days of SPY bars + 30 days of option chain + ~30 ATM contract bars). Cached reruns are ~5 min.

In [ ]:
!python -m iron_condor.cli --sweep

## 4. Top configs

In [ ]:
import pandas as pd
summary = pd.read_csv('results/sweep_summary.csv')
summary.head(20)

## 5. Timing analysis — how long do winners take?
Use this to tune the time-stop. If winners typically resolve in 10 min, a 30-min time-stop is wasteful — try 15. If they need 25 min, 15 would kill them.

In [ ]:
from iron_condor.metrics import analyze_timing
trades = pd.read_csv('results/sweep_trades.csv')
print('=== All configs combined ===')
print(analyze_timing(trades))

In [ ]:
# Top-5 configs broken out
for cfg in summary.head(5)['config'].tolist():
    print(f'\n--- {cfg} ---')
    print(analyze_timing(trades[trades['config'] == cfg]))

In [ ]:
# Direction split: do longs perform differently than shorts?
taken = trades[trades['exit_reason'].isin(['profit', 'stop', 'time'])]
if not taken.empty and 'signal_direction' in taken.columns:
    print(taken.groupby('signal_direction').agg(
        trades=('net_pnl', 'count'),
        win_rate=('net_pnl', lambda s: (s > 0).mean()),
        avg_pnl=('net_pnl', 'mean'),
        median_pnl=('net_pnl', 'median'),
    ).round(2))